In [ ]:
import sys,os,re
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from source_code.galdist import galaxy_distribution
from source_code.gwdist import gw_distribution

from itertools import product
from copy      import deepcopy
from time      import time

from scipy.interpolate import interp1d
from scipy.integrate   import trapz

import warnings
warnings.filterwarnings('ignore')

import matplotlib
from matplotlib import rc
from matplotlib.pyplot import cm
from matplotlib.colors import LogNorm

rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}


# Settings

In [ ]:
#Euclid survey specifications, N_gw in [10^5-10^6] according to ET
galaxy_specs = {'fsky': 0.35, 
                'gal_per_arcmin': 30.,
                'sigma_eps': 0.3,  #sigma associated to noise for GC and WL
                'Nbin_ell': 20,
                'lmin': 10,
                'lmax': 1500}

GW_specs = {'fsky': 0.35, 
            'N_gw': 10**5, 
            'sigma_eps_gw': 0.005} #sigma associated to noise (d_L) for GW-WL

analysis_settings = {'Nbin_ell': 20,
                     'lmin': 10,
                     'lmax': 1500}

use_obs  = ['GC','WL','GWC']#, 'GWC']


## Preliminary calculations

Setting up some quantities that will be used later

In [ ]:
lmin = np.log10(analysis_settings['lmin'])
lmax = np.log10(analysis_settings['lmax'])
N    = analysis_settings['Nbin_ell']

ell_lims = np.logspace(lmin,lmax,N) #creation of array-> N bin log spaced
ells     = np.array([int(ell) for ell in 0.5*(ell_lims[:-1]+ell_lims[1:])]) 
#evaluation of middle points of each bin 
deltas   = (ell_lims[1:]-ell_lims[:-1]) #evaluation of the amplitude of each bin

# Fiducial cosmology and CAMB settings

In [ ]:
#Cosmological parameters describing the LCDM Universe
fiducial = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            #'A_IA': 1.72,
            #'eta_IA': -0.41,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}
MG_params={'MG_flag': 1,
           'pure_MG_flag': 2,
           'musigma_par': 1,
           'DE_model': 0,
           'sigma0': 0.61,
           'mu0': 0.64}
fiducial.update(MG_params)


#camb_path = '/Users/chiaradeleo/myenv/lib/python3.12/site-packages'


# Galaxy distributions (external)

These cells create the galaxy distribution object that is needed by the obs computation.
We save all this info in a dictionary that will 

In [ ]:
distributions = {}

In [ ]:
if 'GC' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gc = len(bin_lims)-1
    
    distributions['GC'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_gc,
                           'zmean': bin_mids}
    
if 'WL' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_wl = len(bin_lims)-1
    
    distributions['WL'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_wl,
                           'zmean': bin_mids}
    
if 'GWC' in use_obs:
    
    gwdist  = gw_distribution('ET-5')
    
    
    bin_lims = gwdist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gwc = len(bin_lims)-1
    
    distributions['GWC'] = {'dist': gwdist.gwdict['binned_dist'],
                           'Nbins': Nbins_gwc,
                           'zmean': bin_mids}
if 'GWWL' in use_obs:
    gwdist  = gw_distribution('ET-5')
    
    
    bin_lims = gwdist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gwl = len(bin_lims)-1
    
    distributions['GWWL'] = {'dist': gwdist.gwdict['binned_dist'],
                           'Nbins': Nbins_gwl,
                           'zmean': bin_mids}

In [ ]:
data_root = './mock_data/'


if MG_params: 
    if MG_params['MG_flag']==0:
        data_root = data_root+'LCDM_test_'
    else:
        data_root = data_root+f'MGflag_{MG_params["MG_flag"]}_test_'
        
if 'GC' in use_obs and 'WL' in use_obs:
    data_root = data_root+'gal'
elif 'GC' in use_obs:
    data_root = data_root+'GC'
elif 'WL' in use_obs:
    data_root = data_root+'WL'
if 'GWWL' in use_obs and 'GWC' in use_obs:
        data_root = data_root+'GWs_Nbin'+str(distributions['GWWL']['Nbins'])
elif 'GWC' in use_obs:
        data_root = data_root+'GWC'+'GWs_Nbin'+str(distributions['GWC']['Nbins'])
elif 'GWWL' in use_obs:
        data_root = data_root+'GWWL'+'GWs_Nbin'+str(distributions['GWWL']['Nbins'])
print(data_root)


In [ ]:
obs_list = []
for obs in use_obs:
    if obs == 'GC':
        obs_list.append('G')
    elif obs == 'WL':
        obs_list.append('L')
    elif obs == 'GWWL':
        obs_list.append('WL')
    elif obs == 'GWC':
        obs_list.append('WC')

Nbins   = {new_obs: distributions[obs]['Nbins'] for new_obs,obs in zip(obs_list,use_obs)}
maxbins = max(list(Nbins.values()))



# Observables computation

In [ ]:
from source_code.compute_obs_sources import get_obs
extra={}

In [ ]:
settings={'camb_path': '/Users/chiaradeleo/myenv/lib/python3.9/site-packages',
         'case': 'simple',
         'calculation': 'CAMB',
          'extra':extra}
calc_obs = get_obs(fiducial,distributions,ells,settings,feedback=True)


# Constructing covariance

**WARNING:** 
- currently setup for galaxies only
- assumes Gaussian covariance

In [ ]:
from source_code.covariance_utils import covariance_einsum

Nell = {k: [0.]*len(ells) for k in calc_obs.Cls.columns}


for obs in obs_list:
    for i in range(1,Nbins[obs]+1):
        if obs == 'G':
            ngalbin = (galaxy_specs['gal_per_arcmin']/Nbins[obs])*3600*(180/np.pi)**2
            Nell['{}{}x{}{}'.format(obs,i,obs,i)] = [(1/ngalbin)]*len(ells)
        elif obs == 'L':
            ngalbin = (galaxy_specs['gal_per_arcmin']/Nbins[obs])*3600*(180/np.pi)**2
            Nell['{}{}x{}{}'.format(obs,i,obs,i)] = [(galaxy_specs['sigma_eps']**2/(2*ngalbin))]*len(ells)
        elif obs == 'WL':
            ngwcbin = GW_specs['N_gw']/Nbins[obs]
            Nell['{}{}x{}{}'.format(obs,i,obs,i)] = [(GW_specs['sigma_eps_gw']**2/ngwcbin)]*len(ells)
        elif obs == 'WC':
            ngwcbin = GW_specs['N_gw']/Nbins[obs]
            Nell['{}{}x{}{}'.format(obs,i,obs,i)] = [(1/ngwcbin)]*len(ells)

fsky      = galaxy_specs['fsky']
Delta_ell = deltas

Nobs = len(use_obs)
err_for_cov = np.zeros((Nobs,Nobs,len(ells),maxbins,maxbins))
cls_for_cov = np.zeros((Nobs,Nobs,len(ells),maxbins,maxbins))





In [ ]:
for o1,obs1 in enumerate(obs_list):
    for o2,obs2 in enumerate(obs_list):
        for i in range(Nbins[obs1]):
            for j in range(Nbins[obs2]):
                if obs1 == obs2 and j<i:
                    Nell[obs1+str(i+1)+'x'+obs2+str(j+1)] = Nell[obs1+str(j+1)+'x'+obs2+str(i+1)]
                    calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)] = calc_obs.Cls[obs1+str(j+1)+'x'+obs2+str(i+1)]
                    for ell_ind,ell in enumerate(ells):
                        err_for_cov[o1,o2,ell_ind,i,j] = Nell[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]
                        cls_for_cov[o1,o2,ell_ind,i,j] = calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]

covmat_diag = covariance_einsum(cls_for_cov,err_for_cov,fsky,ells,Delta_ell,return_only_diagonal_ells=True)

# Creating dataset

## Packing covmat

In [ ]:
def split_num(s):
    head = s.rstrip('0123456789')
    tail = s[len(head):]
    return head, tail

In [ ]:
import time
start = time.time()
cols = {}
for o1,obs1 in enumerate(obs_list):
    cols[obs1] = [obs1+'{}x'.format(i)+obs1+'{}'.format(j) for i in range(1,Nbins[obs1]+1) for j in range(i,Nbins[obs1]+1)] 
    for o2,obs2 in enumerate(obs_list):
        if o2>o1:
            cols[obs1+'x'+obs2] = [obs1+'{}x'.format(i)+obs2+'{}'.format(j) for i in range(1,Nbins[obs1]+1) for j in range(1,Nbins[obs2]+1)]

all_cols2 = []

for obscomb,columns in cols.items():
    all_cols = all_cols2 + columns



In [ ]:
str_to_ind = {obs: ind for ind,obs in enumerate(obs_list)}

covmat_dict = {}

for ellind,ell in enumerate(ells):
    
    packed_covmat = pd.DataFrame(columns=all_cols,index=all_cols,dtype='float')
    
    for ind1,col in enumerate(all_cols):
        bin1,bin2 = re.split('x',col)
        oi1,i1 = split_num(bin1)
        oj1,j1 = split_num(bin2)
    
        for ind2,row in enumerate(all_cols):
            bin1,bin2 = re.split('x',row)
            oi2,i2 = split_num(bin1)
            oj2,j2 = split_num(bin2)
            
            packed_covmat.at[row,col] = covmat_diag[str_to_ind[oi1],str_to_ind[oj1],str_to_ind[oi2],str_to_ind[oj2],
                                                       ellind,int(i1)-1,int(j1)-1,int(i2)-1,int(j2)-1]
            
            packed_covmat.index = packed_covmat.columns
            #print(list(packed_covmat.columns))

            
    covmat_dict[str(int(ell))] = packed_covmat

In [ ]:
plot_cov=False
if plot_cov==True:
    for ell in ells:
        plt.figure()
        plt.title(r'Covariance matrix at $\ell={}$'.format(int(ell)))
        sb.heatmap(covmat_dict[str(int(ell))],norm=LogNorm());

## Creating final Cls

In [ ]:
def get_realization(fiducial,covmats):

    noisy_cls = fiducial.copy()
    cols = [col for col in fiducial.columns if col != 'ells']

    for ind,ell in enumerate(fiducial['ells']):
        covmat = covmats[str(int(ell))]

        means  = [fiducial[col][ind] for col in cols]
        sample = np.random.multivariate_normal(means,covmat)
        
        for col_ind,col in enumerate(cols):
            noisy_cls.at[ind,col] = sample[col_ind]

    return noisy_cls

In [ ]:
noiseless_cls = pd.DataFrame(columns=['ells']+all_cols)

noiseless_cls['ells'] = [int(ell) for ell in ells]

for col in all_cols:
    noiseless_cls[col] = calc_obs.Cls[col]
    
noisy_cls = get_realization(noiseless_cls,covmat_dict)

## Some testing plots

In [ ]:
diag_error = noiseless_cls.copy()

for ind,ell in enumerate(ells):
    for col in all_cols:
        diag_error.at[ind,col] = np.sqrt(covmat_dict[str(int(ell))].at[col,col])

## Saving to file if requested

In [ ]:
if data_root != '':
    noiseless_cls.to_csv(data_root+'_Cls_noiseless.dat',sep='\t',header=True)
    noisy_cls.to_csv(data_root+'_Cls_noisy.dat',sep='\t',header=True)
    np.save(data_root+'_source_distribution.npy',distributions)
    np.save(data_root+'_covmat.npy',covmat_dict)